In [1]:
import torch
import os, sys

import polars as pl

sys.path.insert(0, os.path.abspath(".."))

MAC_DIR = '/Users/igwanhyeong/PycharmProjects/paper_research/'
WINDOW_DIR = 'C:/Users/USER/PycharmProjects/paper_research/'

if sys.platform == 'win32':
    DIR = WINDOW_DIR
    print(torch.cuda.is_available())
    print(torch.cuda.device_count())
    print(torch.version.cuda)
    print(torch.__version__)
    print(torch.cuda.get_device_name(0))
    print(torch.__version__)
else:
    DIR = MAC_DIR

In [2]:
marked_df = pl.read_parquet(DIR + 'sample_data/marked_target_df.parquet')

int(marked_df.select((pl.col("mark").max() + 1).alias("k")).item())


8

In [3]:
from torch.utils.data import DataLoader
from data_loader.event_seq_data_module import RMTPPDataset, time_split_events, collate_next_event, \
    RMTPPWeekLookbackDataset, collate_week_lookback
from models.RMTPPs.RMTPP import RMTPPConfig
from utils.training import TrainingConfig, train_rmtpp

training_config = TrainingConfig(
    lookback = 3,
    batch_size = 256,
    max_seq_len = 64,
    lr = 1e-3,
    epochs = 30,
    val_ratio = 0.2,
    device = 'cuda' if torch.cuda.is_available() else 'cpu',
    lambda_dt = 1.0,
)
train_events, val_events = time_split_events(marked_df, val_ratio = training_config.val_ratio)

train_ds = RMTPPDataset(train_events, lookback = training_config.lookback)
val_ds = RMTPPDataset(val_events, lookback = training_config.lookback)

train_loader = DataLoader(train_ds, batch_size = training_config.batch_size, shuffle = True, num_workers = 0, collate_fn = collate_next_event, drop_last = True)
val_loader = DataLoader(val_ds, batch_size = training_config.batch_size, shuffle = False, num_workers = 0, collate_fn = collate_next_event)
print("event concept: train_events rows:", train_events.height)
print("event concept: val_events rows  :", val_events.height)
print("event concept: train_ds len:", len(train_ds))
print("event concept: val_ds len  :", len(val_ds))


training_config = TrainingConfig(
    lookback = 52,
    batch_size = 256,
    max_seq_len = 64,
    lr = 1e-3,
    epochs = 30,
    val_ratio = 0.2,
    device = 'cuda' if torch.cuda.is_available() else 'cpu',
    lambda_dt = 1.0,
)
train_ds = RMTPPWeekLookbackDataset(
        marked_df,
        lookback_weeks=training_config.lookback,  # 새 파라미터
        max_seq_len=training_config.max_seq_len,
        val_ratio=training_config.val_ratio,
        mode="train",
    )
val_ds = RMTPPWeekLookbackDataset(
    marked_df,
    lookback_weeks=training_config.lookback,
    max_seq_len=training_config.max_seq_len,
    val_ratio=training_config.val_ratio,
    mode="val",
)

train_loader = DataLoader(train_ds, batch_size = training_config.batch_size, shuffle=True, collate_fn = collate_week_lookback, drop_last=True)
val_loader = DataLoader(val_ds, batch_size = training_config.batch_size, shuffle=False, collate_fn = collate_week_lookback)

print("ts_concept: train_events rows:", train_events.height)
print("ts_concept: val_events rows  :", val_events.height)
print("ts_concept: train_ds len:", len(train_ds))
print("ts_concept: val_ds len  :", len(val_ds))

event concept: train_events rows: 161046
event concept: val_events rows  : 81842
event concept: train_ds len: 79826
event concept: val_ds len  : 12396
ts_concept: train_events rows: 161046
ts_concept: val_events rows  : 81842
ts_concept: train_ds len: 161046
ts_concept: val_ds len  : 58455


In [4]:
from models.RMTPPs.RMTPP import RMTPPConfig
from utils.training import TrainingConfig, train_rmtpp


K = 8

training_config = TrainingConfig(
    lookback = 52,
    batch_size = 256,
    lr = 1e-3,
    epochs = 30,
    val_ratio = 0.2,
    device = 'cuda' if torch.cuda.is_available() else 'cpu',
    lambda_dt = 1.0,
)

rmtpp_config = RMTPPConfig(
    num_marks = K + 1,
    mark_emb_dim = 32,
    rnn_hidden_dim = 128,
    rnn_type = 'gru',
    dropout = 0,
    eps = 1e-8,
    w_min = 1e-3,
    exp_clamp = 300.00
)

model, info = train_rmtpp(
    marked_df,
    rmtpp_config = rmtpp_config,
    training_config = training_config,
)

model.eval().to(training_config.device)
info

[Epoch 01] train_loss=33.73680327 | val_acc=0.55297237 val_dt_mae=32.78395980 | val_dt_rmse=45.43370265 | total=58455 | correct=32324
[Epoch 02] train_loss=5.11163320 | val_acc=0.56099564 val_dt_mae=33.81554537 | val_dt_rmse=51.30145420 | total=58455 | correct=32793
[Epoch 03] train_loss=4.77816257 | val_acc=0.56280900 val_dt_mae=32.23733921 | val_dt_rmse=46.56224800 | total=58455 | correct=32899
[Epoch 04] train_loss=4.62572592 | val_acc=0.56369857 val_dt_mae=34.01242052 | val_dt_rmse=53.48634156 | total=58455 | correct=32951
[Epoch 05] train_loss=4.64491437 | val_acc=0.56386964 val_dt_mae=33.72971403 | val_dt_rmse=52.88411448 | total=58455 | correct=32961
[Epoch 06] train_loss=4.63531224 | val_acc=0.56467368 val_dt_mae=33.12884695 | val_dt_rmse=51.43165988 | total=58455 | correct=33008
[Epoch 07] train_loss=4.56389257 | val_acc=0.56286032 val_dt_mae=32.19798151 | val_dt_rmse=47.83378255 | total=58455 | correct=32902
[Epoch 08] train_loss=4.56934554 | val_acc=0.55974681 val_dt_mae=35.

{'best_score': 0.24088050478533168}

In [ ]:
def build_rep_qty_table(marked_df: pl.DataFrame, K: int, tail_bins = (None, None)) -> pl.DataFrame:
    '''
    :param marked_df:
           Schema([('oper_part_no', String),
                   ('demand_dt', Int64),
                   ('seq', UInt32),
                   ('delta_t', Int32),
                   ('demand_qty', Float64),
                   ('z', Float64),
                   ('mark', Int32)])
    :param K: 8
    :param tail_bins: (K-2, K-1) 같은 tail bin id를 넣어주면 해당 bin은 p90으로 대체
    :return: DataFrame[mark: int, rep_qty: float]
    '''

    base = (
        marked_df
          .group_by('mark')
          .agg([
            pl.col('demand_qty').median().alias('rep_qty_median'),
            pl.col('demand_qty').quantile(0.90, 'nearest').alias('rep_qty_p90'),
            pl.len().alias('cnt'),
          ])
          .sort('mark')
    )

    b1, b2 = tail_bins
    if b1 is None: b1 = K - 2
    if b2 is None: b2 = K - 1

    # tail bin은 p90을 사용, 나머지는 median
    rep = (
        base
          .with_columns(
            pl.when(pl.col('mark').is_in([b1, b2]))
              .then(pl.col('rep_qty_p90'))
              .otherwise(pl.col('rep_qty_median'))
              .alias('rep_qty')
          )
          .select(['mark', 'rep_qty', 'cnt'])
    )

    return rep
import numpy as np
def rep_qty_to_tensor(rep_table: pl.DataFrame, K: int):
    rep = np.zeros((K,), dtype=np.float32)
    rows = rep_table.select(['mark', 'rep_qty']).to_numpy()
    for m, q in rows:
        rep[int(m)] = float(q)
    return rep

# K = 8
# rep_table = build_rep_qty_table(marked_df, K=K, tail_bins=(K-2, K-1))
# rep_qty = rep_qty_to_tensor(rep_table, K=K)  # shape (K,)
# rep_table

In [ ]:
import numpy as np
import torch


@torch.no_grad()
def simulate_horizon_grid_with_user_wrapper(
    model,
    dt_hist: torch.Tensor,     # (L,) float
    mk_hist: torch.Tensor,     # (L,) long
    H: int,
    rep_qty: np.ndarray,       # (K,) float32
    n_sims: int = 200,
    max_events: int = 200,
    use_sampling: bool = True,
    min_dt: float = 1.0,
    device: str = "cpu",
    week_index_rule: str = "ceil_minus_1",  # "ceil_minus_1" | "floor"
):
    """
    User wrapper I/O에 맞춘 Horizon 시뮬레이션 + 주차 그리드 복원.

    Assumption: model.predict_next(marks, dts) -> (y_hat, dt_hat, prob)
      - marks: [1, L] long
      - dts:   [1, L] float
      - y_hat: [1] long (argmax)
      - dt_hat:[1] float (sample)
      - prob:  [1, K] float (softmax already applied)

    Returns:
      mean_grid: (H,) float32
      all_grids: (n_sims, H) float32
    """

    if week_index_rule not in ("ceil_minus_1", "floor"):
        raise ValueError("week_index_rule must be 'ceil_minus_1' or 'floor'")

    rep_qty_t = torch.tensor(rep_qty, dtype=torch.float32, device=device)

    dt_hist = dt_hist.to(device).float()
    mk_hist = mk_hist.to(device).long()

    all_grids = torch.zeros((n_sims, H), dtype=torch.float32, device=device)

    for s in range(n_sims):
        # rolling history
        dt_seq = dt_hist.clone()
        mk_seq = mk_hist.clone()

        t = 0.0
        grid = torch.zeros((H,), dtype=torch.float32, device=device)

        for _ in range(max_events):
            # ---- 1) next dt / mark distribution ----
            # user wrapper expects (marks, dts)
            y_hat, dt_hat, prob = model.predict_next(
                mk_seq.unsqueeze(0),  # [1, L]
                dt_seq.unsqueeze(0),  # [1, L]
            )

            dt_next = float(dt_hat.squeeze().item())

            # dt 안정화: 최소 1주 이상
            if dt_next < min_dt:
                dt_next = min_dt

            # 시간 누적
            t = t + dt_next
            if t > H:
                break

            # ---- 2) mark 선택 (샘플링/결정) ----
            # prob: [1, K] (softmax applied)
            if use_sampling:
                mk_next = int(torch.multinomial(prob[0], 1).item())
            else:
                mk_next = int(y_hat.squeeze().item())  # wrapper argmax 사용

            # ---- 3) time -> weekly grid ----
            if week_index_rule == "ceil_minus_1":
                # dt가 "주 단위"라는 해석에 가장 직관적:
                # dt=1이면 horizon 첫 칸(index 0)에 반영
                w = int(np.ceil(t) - 1)
            else:
                # 연속시간 해석을 그대로 쓰는 경우
                w = int(np.floor(t))

            if 0 <= w < H:
                grid[w] += rep_qty_t[mk_next]

            # ---- 4) roll history (append dt_next, mk_next) ----
            dt_seq = torch.cat([dt_seq[1:], torch.tensor([dt_next], device=device)], dim=0)
            mk_seq = torch.cat([mk_seq[1:], torch.tensor([mk_next], device=device)], dim=0)

        all_grids[s] = grid

    mean_grid = all_grids.mean(dim=0)
    return mean_grid.detach().cpu().numpy(), all_grids.detach().cpu().numpy()


# -----------------------------
# (옵션) mean_grid -> int 벡터로 변환
# -----------------------------
def grid_to_int_list(mean_grid: np.ndarray, rounding: str = "round") -> list[int]:
    """
    rounding: "round" | "floor" | "ceil"
    """
    if rounding == "round":
        return np.rint(mean_grid).astype(int).tolist()
    if rounding == "floor":
        return np.floor(mean_grid).astype(int).tolist()
    if rounding == "ceil":
        return np.ceil(mean_grid).astype(int).tolist()
    raise ValueError("rounding must be 'round', 'floor', or 'ceil'")

In [ ]:
from utils.rntpp_wrapper import RMTPPWrapper

# 1) rep_qty 준비
K = 12
rep_table = build_rep_qty_table(marked_df, K=K, tail_bins=(K-2, K-1))
rep_qty = rep_qty_to_tensor(rep_table, K=K)

# 2) 특정 자재 최근 L개 이벤트
L = 30
part = "0001-1002"

part_ev = (
    marked_df
    .filter(pl.col("oper_part_no") == part)
    .sort("seq")
)

# 이벤트 수가 부족하면 L을 줄여야 함
dt_hist = torch.tensor(part_ev.tail(L)["delta_t"].to_numpy(), dtype=torch.float32)
mk_hist = torch.tensor(part_ev.tail(L)["mark"].to_numpy(), dtype=torch.long)

# 3) 예측
H = 13
wrapped = RMTPPWrapper(model)

mean_grid, all_grids = simulate_horizon_grid_with_user_wrapper(
    model=wrapped,
    dt_hist=dt_hist,
    mk_hist=mk_hist,
    H=H,
    rep_qty=rep_qty,
    n_sims=200,
    use_sampling=True,
    device="cuda",
    week_index_rule="ceil_minus_1",  # ✅ 권장 (오프바이원 방지)
)

mean_grid  # (13,) float
pred_int = grid_to_int_list(mean_grid, rounding="round")
pred_int